# RVC — Giải thích chi tiết **từng bước** khi train **một giọng người dùng**

Tài liệu này **không thay** notebook chạy lệnh (`RVC_training_standalone.ipynb`), mà **bổ sung ý nghĩa** mỗi bước: đầu vào/ra là gì, tại sao cần, và nó được gọi từ đâu trong code `rvc_standalone`.

**Tham chiếu mã nguồn:**
- Điều phối: `training_pipeline/steps.py` (`step_preprocess`, `step_extract_f0_and_features`, `step_train`, `step_train_index`, `step_extract_small_weights`).
- Preprocess: `infer/modules/train/preprocess.py`.
- F0 + Hubert: `infer/modules/train/extract/` (ví dụ `extract_f0_print.py`, `extract_f0_rmvpe.py`, `extract_feature_print.py`).
- Huấn luyện: `infer/modules/train/train.py`.

---

## Tổng quan: bạn đang train **cái gì**?

RVC (Retrieval-based Voice Conversion) khi **train** một giọng đích **chủ yếu** học một mạng sinh âm (**Generator `G`**) — và đi kèm **Discriminator `D`** trong gan training — để:

1. Nhận đặc trưng nội dung (content) gần giống **Hubert** từ giọng bất kỳ.
2. Kết hợp **F0** (cao độ / ngữ điệu) nếu model có nhánh F0.
3. Phát ra **waveform** (hoặc representation trung gian) **giống chất giọng / timbre** của dữ liệu bạn đưa vào.

Bước **FAISS index** (sau train) **không** học thêm trọng số mạng; nó chỉ lưu **kho vector Hubert** của **chính giọng huấn luyện** để lúc infer **lấy lại** các đoạn embedding tương tự (retrieval), giúp giọng ổn định và gần dữ liệu gốc hơn.

```mermaid
flowchart LR
  subgraph data_prep [Chuẩn bị dữ liệu]
    W[Thư mục .wav gốc]
    P[Preprocess]
    GT[0_gt_wavs]
    S16[1_16k_wavs]
  end
  subgraph acoustic [Trích xuất cho train]
    F0[2a_f0 / 2b-f0nsf]
    HU[3_feature768 hoặc 256]
  end
  subgraph learn [Học tham số]
    TR[train.py: G + D]
  end
  subgraph optional [Tuỳ chọn infer]
    IDX[FAISS index]
    SM[extract_small_model]
  end
  W --> P --> GT
  P --> S16
  GT --> F0
  S16 --> HU
  GT --> TR
  F0 --> TR
  HU --> TR
  HU --> IDX
  TR --> SM
```

---

## Chuẩn bị từ phía **người dùng** (trước khi chạy pipeline)

| Việc cần làm | Lý do |
|--------------|--------|
| Đặt nhiều file **`.wav`** (giọng đích — người bạn muốn **bắt chước** sau này) vào **`trainset_dir`** | Đây là **nguồn học timbre** duy nhất cho G/D. |
| **Dài đủ** (thực tế thường **nhiều phút — hàng chục phút** càng tốt, miễn chất lượng tốt) | Ít dữ liệu → dễ overfit / méo tiếng. |
| Giọng **rõ**, ít nhiễu, **ít nhạc nền** | Giảm artifact và lệch F0. |
| Có **`logs/mute/`** (bộ template im lặng) | `filelist.txt` luôn **thêm** vài mẫu “mute” cân bằng batch — xem `steps._write_filelist`. |
| Tải **`hubert_base.pt`**, **pretrained G/D**, **`rmvpe.pt`** (nếu dùng RMVPE cho F0) | Preprocess không cần Hubert; bước feature và train **cần**. |

**`experiment_name`:** chỉ là tên thư mục `logs/<experiment_name>/` — mọi artifact của lần train này nằm gọn trong đó.

---

## Bước 1 — **Preprocess** (`step_preprocess` → `preprocess.py`)

### Mục đích

Đưa **các file lộn xộn** trong thư mục dataset thành **chuỗi clip ngắn, chuẩn hóa**, sẵn sàng cho F0 và Hubert.

### Xử lý chính (trong `preprocess.py`, class `PreProcess`)

1. **Đọc audio** bằng `load_audio(path, sr)` với `sr` = **32000 / 40000 / 48000** tùy bạn chọn `sample_rate_label` (khớp **40k**, **48k**, **32k**).
2. **High-pass** (lọc thông cao ~48 Hz) để bỏ phần bass rất thấp có thể làm nhiễu slicing.
3. **`Slicer`**: cắt theo **khoảng lặng / năng lượng** — tách bài dài thành **đoạn có tiếng nói/hát**, bỏ quãng im lặng dài.
4. Trên mỗi đoạn sau slicer, cắt tiếp theo **cửa sổ thời gian** `per` giây (mặc định từ `config.preprocess_per`, thường ~3.7s) có **overlap** — tạo nhiều clip ngắn trùng lặp nhẹ → **nhiều mẫu train**.
5. **`norm_write`**: chuẩn hoá biên độ (tránh clip cực đại > 2.5 thì **bỏ** đoạn — coi là hỏng), scale về mức an toàn rồi ghi file.

### Thư mục sinh ra

| Thư mục | Sample rate | Vai trò |
|---------|-------------|---------|
| **`logs/<exp>/0_gt_wavs/`** | **Đúng `sr` đã chọn** (vd. 40 kHz) | **Ground truth** cho train — khớp với config vocoder / sampling của model. |
| **`logs/<exp>/1_16k_wavs/`** | **16 kHz** | Dùng cho **Hubert** (HuBERT/ContentVec thường làm việc 16 kHz). |

Tên file kiểu **`0_0.wav`, `0_1.wav`**: tiền tố **`0`** = chỉ mục file trong `trainset_dir` (sorted); hậu tố = clip thứ mấy sau slicing.

### Tham số bạn hay chỉnh

- **`num_processes`**: song song hóa bao nhiêu worker cho bước này (CPU).
- **`sample_rate_label`**: phải **thống nhất** suốt pipeline (40k, 48k, 32k) và khớp **pretrained** trong `assets/pretrained_v2/…`.

---

## Bước 2 — **Trích F0** (nếu `if_f0=True`)

### Mục đích

**F0** (fundamental frequency) mô tả **cao độ cơ bản** theo thời gian: lên xuống **giống giai điệu / ngữ điệu**. Model có nhánh F0 (`if_f0=1`) **cần** các file này để học **không** ăn trộm luôn cả nội dung lẫn pitch trong một vector.

### Code được gọi (`step_extract_f0_and_features`)

- **`f0_method != "rmvpe_gpu"`**: `extract_f0_print.py` — F0 cổ điển (pm / harvest / crepe / rmvpe tùy cấu hình).
- **`rmvpe_gpu`**: `extract_f0_rmvpe.py` với GPU; hoặc `extract_f0_rmvpe_dml` cho DirectML.

### Thư mục / file

| Đường dẫn | Nội dung |
|-----------|----------|
| **`2a_f0/*.wav.npy`** | Chuỗi F0 (hoặc biến thể log) theo frame — dùng trong **training / conditioning**. |
| **`2b-f0nsf/*.wav.npy`** | Biến thể **NSF** (non-negative frequency / liên quan sinh sóng) cho nhánh sinh có F0. |

Tên file **khớp stem** với `0_gt_wavs` (cùng `0_0`, `0_1`, …) để sau này ghép **1 dòng trong filelist**.

### Lưu ý

- Nếu **`if_f0=False`** (model non-F0), **bỏ** cả nhánh F0; filelist chỉ cần wav + feature.
- **`rmvpe`** cần **`assets/rmvpe/rmvpe.pt`** (và GPU đủ VRAM) — giống infer.

---

## Bước 3 — **Trích đặc trưng Hubert** (`extract_feature_print.py`)

### Mục đích

Mỗi clip trong **`1_16k_wavs`** được đưa qua **HuBERT (content)** đã tải (`assets/hubert/hubert_base.pt`). Đầu ra là **vector theo thời gian** biểu diễn **nội dung lời nói** (ít phụ thuộc speaker hơn raw spectrogram).

### Thư mục

| Phiên bản | Thư mục | Chiều vector |
|-----------|---------|----------------|
| **v1** | **`3_feature256/`** | 256 |
| **v2** | **`3_feature768/`** | 768 |

Mỗi file **`.npy`**: ma trận `(T, D)` với `D` là 256 hoặc 768.

### Tại sao train **không** “học trực tiếp từ wav”?

Hubert **đóng băng** (hoặc chỉ dùng forward) trong bước này — ta dùng nó như **bộ mã hóa nội dung cố định**. `G` học **ánh xạ** từ (content + F0 + speaker id) → waveform/spectrogram trong kiến trúc RVC.

### Multi-GPU

`step_extract_f0_and_features` có thể chia **theo danh sách GPU** (`gpu_devices_train` dạng `0-1`) — mỗi process xử lý một phần danh sách file.

---

## Bước 4 — **Ghép filelist + config.json** rồi **Train** (`step_train`)

### 4a. `_write_filelist`

Hàm này **duyệt** các thư mục đã có:

- Lấy **giao** tập tên (stem) giữa `0_gt_wavs`, `3_feature*`, và (nếu có F0) `2a_f0`, `2b-f0nsf` — chỉ những file **đủ cặp** mới vào train.
- Mỗi dòng `filelist.txt` là một chuỗi **đường dẫn nối với nhau bằng `|`**: wav ground truth | feature | [f0 files] | **speaker_id**.
- **Thêm 2 dòng “mute”** từ `logs/mute/...` để dataset luôn có mẫu **im lặng** — tránh model “kẹt” khi input gần như không có pitch.
- **Xáo trộn** (`shuffle`) các dòng.
- Nếu chưa có **`logs/<exp>/config.json`**, copy từ `configs/inuse/…` tương ứng (`v1/40k.json` hoặc `v2/48k.json`, v.v.).

### 4b. `train.py`

- Nạp **`G`** từ **pretrained** (`f0G40k.pth` / tương đương) và **`D`** (`f0D40k.pth`).
- Huấn luyện **GAN-style**: `G` cố sinh âm giống thật; `D` phân biệt thật/giả.
- Lưu **`G_*.pth`**, **`D_*.pth`** theo `save_every_epoch` / `total_epochs`.

### **`G` vs `D`** (tóm tắt)

| Thành phần | Vai trò | Khi infer bạn dùng |
|------------|---------|---------------------|
| **G (Generator)** | Sinh **giọng đích** từ Hubert + F0 + spk | **Bắt buộc** — đây là model chở timbre. |
| **D (Discriminator)** | Học phân loại **thật/giả** để ép G tốt hơn | **Không** cần trong infer — chỉ dùng lúc train. |

### Tham số hay gặp

- **`total_epochs`**: số vòng qua dataset; quá ít → chưa “bám” giọng; quá nhiều + ít data → overfit.
- **`batch_size`**: lớn → nhanh hơn nhưng tốn VRAM.
- **`save_only_latest`**: chỉ giữ checkpoint mới nhất hay giữ nhiều file `G_*.pth`.

---

## Bước 5 — **FAISS index** (`step_train_index`) — *retrieval*

### Mục đích **sau train**

Gom **toàn bộ** vector trong `3_feature256` hoặc `3_feature768` thành ma trận lớn, rồi train một chỉ mục **FAISS** (ở đây dạng **IVF + Flat**) để lúc infer **tìm nhanh** các vector HuBERT **gần giống** vector hiện tại — **trong chính không gian của giọng đã train**.

### Các file

- **`total_fea.npy`**: toàn bộ feature xếp chồng (có thể qua **KMeans 10k tâm** nếu quá lớn — giảm kích thước trước khi train index).
- **`trained_IVF...index`**: index sau bước **`train`** của FAISS.
- **`added_IVF...index`**: sau **`add`** — embedding thật sự được **thêm vào** index (file hay dùng cho infer: **added**).

### Tham số `n_ivf`

Code: `n_ivf = min(int(16 * sqrt(N)), N // 39)` — số **cụm IVF** tùy quy mô dữ liệu.

### `outside_index_root`

Nếu set trong `.env`, bản `added_*.index` có thể được **copy/symlink** sang `assets/indices/` để infer dễ tìm.

### Có train được **không cần index**?

**Có.** Đặt `skip_index=True` (hoặc bỏ qua bước 4 trong notebook). Infer vẫn chạy với **`index_rate=0`** (không trộn retrieval).

---

## Bước 6 (tuỳ chọn) — **Trích model nhỏ để infer** (`extract_small_model`)

### Mục đích

File `G_2333333.pth` (hoặc tên khác) trong `logs/<exp>/` chứa **toàn bộ tensor** cần cho **huấn luyện** (có thể rất nặng / nhiều metadata). **`extract_small_model`** tạo file **`.pth` nhẹ**, chỉ đủ cho **tab Infer / notebook infer** (`assets/weights/...`).

### Bạn cần nhớ

- Infer thường dùng **bản nhỏ** + (tuỳ chọn) **file `.index`**.
- **Cùng một run train**: `version`, `sample_rate`, `if_f0` phải **khớp** giữa train và infer.

---

## Liên hệ với **infer** (sau khi train xong)

| Train sinh ra | Infer dùng |
|----------------|------------|
| Hubert (luôn có sẵn `hubert_base.pt`) | Trích content từ **audio nguồn** |
| **`G`** (checkpoint hoặc bản nhỏ) | Sinh **giọng đích** |
| **`added_*.index`** | Retrieval — trộn embedding (slider **index rate** trong WebUI) |
| RMVPE (`rmvpe.pt`) | Ước F0 **audio nguồn** nếu chọn thuật toán rmvpe |

Luồng tổng thể infer: **Nguồn A** → Hubert + F0 → (tuỳ chọn) FAISS → **G** → audio **nghe như model đã train**.

---

## Chạy thực tế

Mở và chạy lần lượt: **`RVC_training_standalone.ipynb`** (trong cùng thư mục `rvc_standalone`). Notebook này (**`RVC_training_giai_thich_chi_tiet.ipynb`**) chỉ để **đọc — không bắt buộc chạy ô code**.